<a href="https://colab.research.google.com/github/mrdantzig/nlp-bible-code/blob/master/15%EC%9E%A5_Text%20summarization/%5B15-1%5D%20TextRank%EB%A5%BC%20%EC%9D%B4%EC%9A%A9%ED%95%9C%20%EC%B6%94%EC%B6%9C%20%EC%9A%94%EC%95%BD_%EC%88%98%EC%A0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [15-1] TextRank를 이용한 추출 요약 (수정본)

자연어처리 바이블 15장 실습 [15-1]을 최신 Colab에서 실행되도록 고친 노트북입니다.

**원본 대비 수정 사항**
- 원본은 `gensim.summarization.summarizer.summarize`를 씁니다. 이 모듈은 **gensim 4.0부터 삭제**되어 현재 Colab에서는 `ModuleNotFoundError`가 납니다.
- 같은 TextRank 아이디어(문장 유사도 그래프 + PageRank)를 scikit-learn과 networkx로 직접 구현했습니다. 둘 다 Colab에 기본으로 설치되어 있고, **원본과 같은 4문장**이 추출됩니다.
- 시는 행이 문장부호로 끝나지 않는 경우가 많아 일반 문장 분리기로 나누면 여러 행이 한 문장으로 합쳐집니다. 그래서 **행 단위로 문장을 나누고**, 반복되는 행은 하나로 합칩니다.
- 7번 (추가) 셀에서 `sumy` 라이브러리의 TextRank·LexRank 결과와 비교해 봅니다.

## 1. 원본 코드 (참고용 · gensim 3.x에서만 동작)

```python
from gensim.summarization.summarizer import summarize
print(summarize(text))
```

원본 출력 (기본 `ratio=0.2` → 전체 문장의 20%):
```
And she won't eat her dinner - rice pudding again -
I've promised her dolls and a daisy-chain,
I've promised her sweets and a ride in the train,
And it's lovely rice pudding for dinner again!
```

## 2. 요약할 원문: A. A. Milne의 시 'Rice Pudding'

In [1]:
text = '''Rice Pudding - Poem by Alan Alexander Milne
What is the matter with Mary Jane?
She's crying with all her might and main,
And she won't eat her dinner - rice pudding again -
What is the matter with Mary Jane?
What is the matter with Mary Jane?
I've promised her dolls and a daisy-chain,
And a book about animals - all in vain -
What is the matter with Mary Jane?
What is the matter with Mary Jane?
She's perfectly well, and she hasn't a pain;
But, look at her, now she's beginning again! -
What is the matter with Mary Jane?
What is the matter with Mary Jane?
I've promised her sweets and a ride in the train,
And I've begged her to stop for a bit and explain -
What is the matter with Mary Jane?
What is the matter with Mary Jane?
She's perfectly well and she hasn't a pain,
And it's lovely rice pudding for dinner again!
What is the matter with Mary Jane?'''

## 3. TextRank 직접 구현

1. 행 단위로 문장을 나누고 같은 문장은 하나로 합친다.
2. 각 문장을 TF-IDF 벡터로 바꾸고, 문장 간 코사인 유사도를 구한다.
3. 유사도를 간선 가중치로 하는 그래프에 PageRank를 적용한다 (= TextRank 점수).
4. 점수 상위 (전체 문장 수 × ratio)개를 원문 순서대로 출력한다.

In [2]:
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def split_lines(text):
    """행 단위로 문장을 나누고, 같은 문장은 하나로 합친다."""
    lines = [s.strip() for s in text.split('\n')]
    lines = [s for s in lines if s]
    return lines, list(dict.fromkeys(lines))


def textrank_scores(sents):
    """TF-IDF 코사인 유사도 그래프에 PageRank를 적용한 문장별 점수"""
    tfidf = TfidfVectorizer(stop_words='english').fit_transform(sents)
    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)              # 자기 자신과의 유사도는 제외
    return nx.pagerank(nx.from_numpy_array(sim))


def summarize(text, ratio=0.2):
    """gensim의 summarize(text, ratio=0.2)를 대신하는 함수"""
    lines, sents = split_lines(text)
    scores = textrank_scores(sents)
    k = max(1, int(len(lines) * ratio))   # 전체 문장 수의 ratio만큼
    top = sorted(sorted(scores, key=scores.get, reverse=True)[:k])   # 원문 순서로
    return '\n'.join(sents[i] for i in top)

## 4. 실행 결과 — 원본(gensim)과 같은 4문장

In [3]:
print(summarize(text))

And she won't eat her dinner - rice pudding again -
I've promised her dolls and a daisy-chain,
I've promised her sweets and a ride in the train,
And it's lovely rice pudding for dinner again!


## 5. 문장별 TextRank 점수 확인

반복되는 "What is the matter with Mary Jane?"은 다른 문장과 겹치는 단어가 거의 없어 점수가 낮습니다.

In [4]:
lines, sents = split_lines(text)
scores = textrank_scores(sents)
print(f'전체 {len(lines)}행 → 서로 다른 문장 {len(sents)}개\n')
for i in sorted(scores, key=scores.get, reverse=True):
    print(f'{scores[i]:.4f}  {sents[i]}')

전체 21행 → 서로 다른 문장 12개

0.1323  And it's lovely rice pudding for dinner again!
0.1322  I've promised her dolls and a daisy-chain,
0.1322  I've promised her sweets and a ride in the train,
0.1264  And she won't eat her dinner - rice pudding again -
0.1163  She's perfectly well, and she hasn't a pain;
0.1163  She's perfectly well and she hasn't a pain,
0.0901  Rice Pudding - Poem by Alan Alexander Milne
0.0845  And I've begged her to stop for a bit and explain -
0.0174  What is the matter with Mary Jane?
0.0174  She's crying with all her might and main,
0.0174  And a book about animals - all in vain -
0.0174  But, look at her, now she's beginning again! -


## 6. 생각해 보기: ratio를 바꾸면?

In [5]:
for r in (0.1, 0.2, 0.3):
    print(f'--- ratio = {r} ---')
    print(summarize(text, ratio=r), end='\n\n')

--- ratio = 0.1 ---
I've promised her dolls and a daisy-chain,
And it's lovely rice pudding for dinner again!

--- ratio = 0.2 ---
And she won't eat her dinner - rice pudding again -
I've promised her dolls and a daisy-chain,
I've promised her sweets and a ride in the train,
And it's lovely rice pudding for dinner again!

--- ratio = 0.3 ---
And she won't eat her dinner - rice pudding again -
I've promised her dolls and a daisy-chain,
She's perfectly well, and she hasn't a pain;
I've promised her sweets and a ride in the train,
She's perfectly well and she hasn't a pain,
And it's lovely rice pudding for dinner again!



## 7. (추가) 라이브러리(sumy)와 비교

`sumy`는 TextRank, LexRank 같은 추출 요약 알고리즘을 모아 둔 라이브러리입니다. 행마다 빈 줄을 넣어 한 행이 한 문장으로 인식되게 한 뒤 4문장씩 뽑아 비교합니다. 같은 그래프 기반 방법이라도 유사도 계산 방식이 달라 고르는 문장이 조금씩 다릅니다.

In [ ]:
!pip install -q sumy
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [7]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer

_, sents = split_lines(text)
doc = PlaintextParser.from_string('\n\n'.join(sents), Tokenizer('english')).document

for name, summarizer in [('sumy TextRank', TextRankSummarizer()),
                         ('sumy LexRank', LexRankSummarizer())]:
    print(f'--- {name} ---')
    print('\n'.join(str(s) for s in summarizer(doc, 4)), end='\n\n')

--- sumy TextRank ---
And she won't eat her dinner - rice pudding again -
She's perfectly well, and she hasn't a pain;
And I've begged her to stop for a bit and explain -
She's perfectly well and she hasn't a pain,

--- sumy LexRank ---
And she won't eat her dinner - rice pudding again -
I've promised her sweets and a ride in the train,
And I've begged her to stop for a bit and explain -
And it's lovely rice pudding for dinner again!

